In [6]:
import pandas as pd
from pathlib import Path
from typing import List, Optional

def _normalize_tokens_drop_list(drop_tags: Optional[List[str]]) -> List[str]:
    toks: List[str] = []
    if not drop_tags:
        return toks
    for t in drop_tags:
        if not isinstance(t, str):
            continue
        # support users passing "tag1;tag2" in one string
        parts = [p.strip() for p in t.split(";") if p.strip()]
        toks.extend(parts)
    # lowercased unique
    out: List[str] = []
    seen = set()
    for t in toks:
        tl = t.lower()
        if tl in seen:
            continue
        seen.add(tl)
        out.append(tl)
    return out

def _remove_tag_tokens(tagn_val: str, tags_to_remove_lc: List[str]) -> str:
    """
    Given a semicolon-separated tag string, remove any tokens that match tags_to_remove_lc.
    Matching is case-insensitive and compares full tokens (not substrings).
    """
    if not isinstance(tagn_val, str) or not tagn_val.strip():
        return ""
    tokens = [tok.strip() for tok in tagn_val.split(";")]
    kept = []
    for tok in tokens:
        if not tok:
            continue
        if tok.lower() in tags_to_remove_lc:
            continue
        kept.append(tok)
    # preserve order & de-dup case-insensitively
    seen_lc = set()
    deduped = []
    for tok in kept:
        tl = tok.lower()
        if tl in seen_lc:
            continue
        seen_lc.add(tl)
        deduped.append(tok)
    return ";".join(deduped)

def merge_clean_csvs(
    input_csvs: List[str],
    out_csv: str,
    drop_tags: Optional[List[str]] = None
) -> str:
    # Load and concat
    frames = []
    for p in input_csvs:
        df = pd.read_csv(p)
        frames.append(df)
    if not frames:
        raise ValueError("No input CSVs provided.")
    merged = pd.concat(frames, ignore_index=True, copy=False)

    # Ensure expected columns exist
    expected = ["question","answer","tagn","accepted","score","url"]
    for col in expected:
        if col not in merged.columns:
            merged[col] = ""

    # Deduplicate by (question, answer)
    merged = merged.drop_duplicates(subset=["question","answer"], keep="first").reset_index(drop=True)

    # Remove requested tag tokens from 'tagn'
    tags_to_remove_lc = _normalize_tokens_drop_list(drop_tags)
    if tags_to_remove_lc:
        merged["tagn"] = merged["tagn"].apply(lambda s: _remove_tag_tokens(s, tags_to_remove_lc))

    # Rename 'tagn' -> 'tags'
    merged = merged.rename(columns={"tagn": "tags"})

    # Add reviewer columns (empty)
    if "reviewer_comment" not in merged.columns:
        merged["reviewer_comment"] = ""
    if "reviewer_score" not in merged.columns:
        merged["reviewer_score"] = ""

    # Reorder columns (keep your original order + new ones at the end)
    base_order = ["question","answer","tags","accepted","score","url"]
    other_cols = [c for c in merged.columns if c not in base_order + ["reviewer_comment","reviewer_score"]]
    merged = merged[base_order + other_cols + ["reviewer_comment","reviewer_score"]]

    # Write
    out_path = Path(out_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_path, index=False)
    print(f"✅ merged file -> {out_path} (rows={len(merged)})")
    return str(out_path)

def preview_merge(csv_path: str, name: str = "Merged QA Preview"):
    df = pd.read_csv(csv_path)
    return df


In [7]:
out_path = merge_clean_csvs(
    [
        "/home/biolab-office-1/DATALAB/2025/Genomeer/dataset/04-output/hq/biostars/biostars_metagenomics_tag.csv",
        "/home/biolab-office-1/DATALAB/2025/Genomeer/dataset/04-output/hq/biostars/biostars_metagenomics.csv",
    ],
    "/home/biolab-office-1/DATALAB/2025/Genomeer/dataset/04-output/hq/biostars/full/biostars_db.csv",
    drop_tags=["tag1","tag2"] 
)

✅ merged file -> /home/biolab-office-1/DATALAB/2025/Genomeer/dataset/04-output/hq/biostars/full/biostars_db.csv (rows=9299)


In [8]:
df = preview_merge(out_path)
df.head()

,question,answer,tags,accepted,score,url,reviewer_comment,reviewer_score
0,"Hi everyone, I'm gathering course materials fo...",A lot of material was added to YouTube by many...,"course,teaching;metagenome",False,6,https://www.biostars.org/p/480436/,NaN,NaN
1,"Hello, I ran a binning tool and assessed compl...",Bin completeness and contamination in CheckM a...,"metagenomics,bins,metagenome;metagenome",True,2,https://www.biostars.org/p/9526318/,NaN,NaN
2,Hi! I am working on the benchmarking of differ...,Precision-Recall (PR) curves are mainly useful...,"classification,recall,precision,metagenome;met...",False,3,https://www.biostars.org/p/9531543/,NaN,NaN
3,"Hi, I'm trying to find an up-to-date database ...",NCBI makes SRA metadata files available here: ...,"SRA,NCBI,metagenome,metadata;metagenome",True,2,https://www.biostars.org/p/9535055/,NaN,NaN
4,"Hi all, I have 3 metagenomic assembled phage g...",Sounds like a pretty thorough investigation to...,"metagenome,Spades,SRA,phage,Bowtie;metagenome",False,2,https://www.biostars.org/p/9547029/,NaN,NaN


In [9]:
df.shape

(9299, 8)

(4045, 8)